# Demo — Running Governance as an Ongoing Operation (live AWS)

Governance isn't a one-off project — it runs as a continuous **audit → triage → remediate →
verify** loop across every discipline on the platform. This walkthrough runs `governance_audit()`,
triages findings by signal and severity, demonstrates closing one catalog gap and one Lake
Formation access-control gap, and previews how operational artifacts (dashboards and onboarding
checklists) keep the audit green.

> Runs against live AWS Glue + Lake Formation when provisioned (`lf_backend()` reports `live`);
> falls back to the local enforcement engine offline.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog("trailhead.db")
print("Backend:", lf_backend(), "->", type(gc).__name__)

## 1. Run the unified platform audit
`governance_audit()` rolls up catalog gaps (MP2), contract status (MP3), quality failures (MP4),
lineage coverage (MP6), and Lake Formation access-control gaps on PII tables (MP7):

In [ ]:
findings = gc.governance_audit(contracts_valid=True)
print("Total findings:", len(findings))
display(findings.groupby(["signal", "severity"]).size().reset_index(name="count"))
findings[findings["signal"].isin(["missing_metadata", "access_control_gap"])]

## 2. Close one catalog gap and one access-control gap
Notice the two control-plane signals we can remediate directly in AWS:
1. **`missing_metadata`** in AWS Glue — let's fix `orders` (`missing_owner`).
2. **`access_control_gap`** in AWS Lake Formation — any table carrying `PII` with no Lake Formation
   grant or LF-Tag is flagged as wide open. Right now `customer` is flagged; let's govern it with
   an LF-Tag (`governed=true`) and grant `co_data_steward` access by tag.

*(Watch out in the exercise: once you tag `campaign_membership.cust_ref` as `PII` in Glue,
`campaign_membership` also becomes a PII-bearing table and must be governed in Lake Formation too!)*

In [ ]:
# 1) Fix one catalog gap in Glue
gc.set_table_metadata("orders", owner="orders-team@trailhead.example")

# 2) Bring the `customer` PII table under Lake Formation tag governance
gc.assign_tag("customer", "governed", "true")
gc.grant_by_tag("co_data_steward", "governed", "true")

re_findings = gc.governance_audit(contracts_valid=True)
for sig in ("missing_metadata", "access_control_gap"):
    before = int((findings["signal"] == sig).sum())
    after = int((re_findings["signal"] == sig).sum())
    print(f"{sig:20s}: {before} -> {after}")

## 3. From ad-hoc fixes to standing operational artifacts
Clearing today's control-plane gaps leaves ongoing data SLOs (`quality_violation`,
`consent_conflict`, `missing_lineage`). To keep the platform compliant over time, the exercise
asks you to design two operational artifacts and a synthesis brief:

- **Governance Dashboard Spec** — e.g., a weekly panel tracking:

| Panel / Metric | Source Signal | Target Threshold | Owner | Alert Rule |
|---|---|---|---|---|
| Catalog & LF Control Plane | `missing_metadata` + `access_control_gap` | `0` open findings | Central Governance Lead | Page owner on any `> 0` |

- **New-Domain Onboarding Checklist** — e.g., gate #1 before a table goes live:

| Step | What to do | Governance signal satisfied |
|---|---|---|
| 1. Register table metadata | Set `owner`, `classification`, and ISO-8601 `retention` in Glue | `missing_metadata == 0` (table-level) |

**Takeaway / your turn:** in the exercise you'll clear **all** `missing_metadata` and
`access_control_gap` findings across the platform, write the operations summary, design the
full dashboard spec and 6-step onboarding checklist, and synthesize your findings into a
Mini Architect's Brief.